# refshift — reference-mismatch decoder experiments

CSP+LDA, ShallowConvNet, EEGNet, ATCNet mismatch matrices (no-EA and EA),
full jitter, leave-one-reference-out (LORO), and leave-one-family-out (LOFO).

Set `DATASET` in section B and run top to bottom. Every experiment is cached to
CSV, so re-running resumes instead of recomputing. Each mismatch matrix is
reported two ways: the pooled matrix/family view (for reading structure) and the
**subject-level transfer gap with a bootstrap CI** (the number that goes in the
paper — seeds are averaged within subject, the subject is the unit of inference).

## A — Setup (run once per kernel)

### A1. Install refshift

The lean package ships as a Kaggle dataset. Point `REFSHIFT_SRC` at the folder
that contains `pyproject.toml`, then install it editable.

In [ ]:
import os, subprocess, sys, importlib

# Folder containing pyproject.toml inside your attached Kaggle dataset.
REFSHIFT_SRC = "/kaggle/input/datasets/delhialli/refshift-lean/refshift-lean"
assert os.path.isfile(os.path.join(REFSHIFT_SRC, "pyproject.toml")), \
    f"pyproject.toml not found under {REFSHIFT_SRC}; fix REFSHIFT_SRC"

subprocess.run(["pip", "install", "-q", "-e", f"{REFSHIFT_SRC}[dl]"], check=True)
subprocess.run(["pip", "install", "-q", "moabb", "braindecode"], check=True)
subprocess.run(["pip", "uninstall", "-y", "-q", "mne", "mne-bids"], check=False)
subprocess.run(
    ["pip", "install", "-q", "--no-cache-dir", "mne==1.11.0", "mne-bids>=0.18"],
    check=True,
)

# An editable install adds a .pth read at startup; a kernel that began before
# the install won't see it. Add the src dir to sys.path so this kernel can import.
if REFSHIFT_SRC not in sys.path:
    sys.path.insert(0, REFSHIFT_SRC)
importlib.invalidate_caches()
print("install done")

### A2. Environment setup + Kaggle dataset symlinks

`setup_kaggle_env()` sets MNE_DATA / thread caps and symlinks your attached
datasets into MOABB's cache layout, so nothing downloads. Run BEFORE any load.

In [ ]:
from refshift import setup_kaggle_env
setup_kaggle_env()   # symlinks the attached datasets; idempotent

### A3. Imports, paths, run-or-skip cache

In [ ]:
import time, warnings
import numpy as np
import pandas as pd

from refshift import (
    calibrate_csp_lda,
    run_mismatch, run_mismatch_jitter, run_loro_matrix, run_lofo_matrix,
    REFERENCE_MODES, FAMILIES, reference_modes_for_dataset, canonical_mode_tuple,
    report_matrix, report_families, report_jitter_full, report_loro, report_lofo,
    report_transfer_gap, transfer_gap_ci,
)

import mne
mne.set_log_level("ERROR")
warnings.filterwarnings("ignore")

BASE, CACHE, RESULTS = "/kaggle/working", "/kaggle/working/cache", "/kaggle/working/results"
for d in (CACHE, RESULTS):
    os.makedirs(d, exist_ok=True)


def run_or_skip(name, fn, force=False):
    """Run fn() -> DataFrame, cache to CSV, or reload an existing CSV."""
    path = f"{RESULTS}/{name}.csv"
    if (not force) and os.path.exists(path):
        df = pd.read_csv(path)
        print(f"[skip] {name} (loaded {len(df)} rows)")
        return df
    t0 = time.time()
    df = fn()
    df.to_csv(path, index=False)
    print(f"[done] {name}  rows={len(df)}  {(time.time()-t0)/60:.1f} min")
    return df

print("results dir:", RESULTS)

## B — Config (the only cell you edit)

In [ ]:
DATASET    = "iv2a"        # iv2a | openbmi | cho2017 | dreyer2023 | schirrmeister2017
REFERENCES = None          # None -> dataset-safe default set (drops cz_ref for schirrmeister)
SEEDS      = [0, 1, 2]     # DL runs average these; CSP+LDA is deterministic (seed 0)
TAG        = "v1"
FORCE      = False         # True = ignore cached CSVs and recompute
DL_MAX_EPOCHS, DL_BATCH_SIZE = 200, 32

MODES = (reference_modes_for_dataset(DATASET) if REFERENCES is None
         else canonical_mode_tuple(REFERENCES))
print(f"DATASET={DATASET}  MODES={MODES} ({len(MODES)})  SEEDS={SEEDS}")

## C — Calibration (sanity check, IV-2a only)

Confirms bare CSP+LDA reproduces the MOABB baseline and that a no-op reference
transformer changes nothing.

In [ ]:
if DATASET == "iv2a":
    _, _, passed = calibrate_csp_lda("iv2a", subjects=[1])
    print("calibration passed:", passed)
else:
    print("calibration skipped for", DATASET)

## C2 — Operator invertibility (algebraic, data-free)

Per reference operator: is it canonicalizable by re-referencing
(`contrast_preserving`, H·M = H), are the native channel contrasts still linearly
recoverable from its output (`contrasts_recoverable`), and how well-conditioned is
that recovery (`cond_contrast`, ~1 trivial, large = ill-conditioned). This grounds
the global-vs-spatial claim: global refs collapse to a common reference, while
Laplacians change the coordinate system but (if recoverable, well-conditioned) lose
no contrast information — so a fixed decoder's failure on them is a coordinate
mismatch, not lost information.

In [ ]:
from refshift import contrast_recovery_report
from refshift.preprocess import load_windows
from refshift.datasets import subject_list

# Channel names/order the experiments use (cached after the first load).
_, _, _, _, CH_NAMES = load_windows(DATASET, subject_list(DATASET)[0], cache_dir=CACHE)
inv = contrast_recovery_report(CH_NAMES, modes=MODES)
print(inv.to_string(index=False))
inv.to_csv(f"{RESULTS}/{DATASET}_operator_invertibility_{TAG}.csv", index=False)

### Reporting helper

`report_all` prints the pooled matrix, the family view, and the subject-level
transfer gap with its bootstrap CI. The CI is the paper number.

In [ ]:
def report_all(df, label):
    report_matrix(df, title=label, modes=MODES)
    report_families(df, title=label, modes=MODES)
    report_transfer_gap(df, title=label)

## D — Mismatch matrices, no EA

### CSP+LDA (no EA)

In [ ]:
df_csp_lda = run_or_skip(f"{DATASET}_csp_lda_noEA_{TAG}", lambda: run_mismatch(
    DATASET, model="csp_lda", seeds=[0], reference_modes=REFERENCES, apply_ea=False,
), force=FORCE)
report_all(df_csp_lda, f"CSP+LDA (no EA) — {DATASET}")

### ShallowConvNet (no EA)

In [ ]:
df_shallow = run_or_skip(f"{DATASET}_shallow_noEA_{TAG}", lambda: run_mismatch(
    DATASET, model="shallow", seeds=SEEDS, reference_modes=REFERENCES, apply_ea=False,
    dl_max_epochs=DL_MAX_EPOCHS, dl_batch_size=DL_BATCH_SIZE, cache_dir=CACHE,
), force=FORCE)
report_all(df_shallow, f"ShallowConvNet (no EA) — {DATASET}")

### EEGNet (no EA)

In [ ]:
df_eegnet = run_or_skip(f"{DATASET}_eegnet_noEA_{TAG}", lambda: run_mismatch(
    DATASET, model="eegnet", seeds=SEEDS, reference_modes=REFERENCES, apply_ea=False,
    dl_max_epochs=DL_MAX_EPOCHS, dl_batch_size=DL_BATCH_SIZE, cache_dir=CACHE,
), force=FORCE)
report_all(df_eegnet, f"EEGNet (no EA) — {DATASET}")

### ATCNet (no EA)

In [ ]:
df_atcnet = run_or_skip(f"{DATASET}_atcnet_noEA_{TAG}", lambda: run_mismatch(
    DATASET, model="atcnet", seeds=SEEDS, reference_modes=REFERENCES, apply_ea=False,
    dl_max_epochs=DL_MAX_EPOCHS, dl_batch_size=DL_BATCH_SIZE, cache_dir=CACHE,
), force=FORCE)
report_all(df_atcnet, f"ATCNet (no EA) — {DATASET}")

## E — Mismatch matrices, with EA

EA is fit per block (train and test independently) after referencing. If it
collapses the gap, the shift is second-order covariance geometry, not lost task
information. Run on the two primary decoders; extend to eegnet/atcnet if needed.

### CSP+LDA (EA)

In [ ]:
df_csp_lda_ea = run_or_skip(f"{DATASET}_csp_lda_EA_{TAG}", lambda: run_mismatch(
    DATASET, model="csp_lda", seeds=[0], reference_modes=REFERENCES, apply_ea=True,
), force=FORCE)
report_all(df_csp_lda_ea, f"CSP+LDA (EA) — {DATASET}")

### ShallowConvNet (EA)

In [ ]:
df_shallow_ea = run_or_skip(f"{DATASET}_shallow_EA_{TAG}", lambda: run_mismatch(
    DATASET, model="shallow", seeds=SEEDS, reference_modes=REFERENCES, apply_ea=True,
    dl_max_epochs=DL_MAX_EPOCHS, dl_batch_size=DL_BATCH_SIZE, cache_dir=CACHE,
), force=FORCE)
report_all(df_shallow_ea, f"ShallowConvNet (EA) — {DATASET}")

## F — Full jitter (DL only)

Train one net with per-sample reference jitter (uniform over operators), test on
every reference. Small spread = the model learned to be reference-invariant.
ShallowConvNet is primary; eegnet/atcnet are optional robustness runs.

### Full jitter — ShallowConvNet

In [ ]:
df_shallow_jit = run_or_skip(f"{DATASET}_shallow_jitter_{TAG}", lambda: run_mismatch_jitter(
    DATASET, model="shallow", condition="full", seeds=SEEDS, reference_modes=REFERENCES,
    dl_max_epochs=DL_MAX_EPOCHS, dl_batch_size=DL_BATCH_SIZE, cache_dir=CACHE,
), force=FORCE)
report_jitter_full(df_shallow_jit, title=f"Full jitter — ShallowConvNet — {DATASET}", modes=MODES)

### Full jitter — EEGNet

In [ ]:
df_eegnet_jit = run_or_skip(f"{DATASET}_eegnet_jitter_{TAG}", lambda: run_mismatch_jitter(
    DATASET, model="eegnet", condition="full", seeds=SEEDS, reference_modes=REFERENCES,
    dl_max_epochs=DL_MAX_EPOCHS, dl_batch_size=DL_BATCH_SIZE, cache_dir=CACHE,
), force=FORCE)
report_jitter_full(df_eegnet_jit, title=f"Full jitter — EEGNet — {DATASET}", modes=MODES)

### Full jitter — ATCNet

In [ ]:
df_atcnet_jit = run_or_skip(f"{DATASET}_atcnet_jitter_{TAG}", lambda: run_mismatch_jitter(
    DATASET, model="atcnet", condition="full", seeds=SEEDS, reference_modes=REFERENCES,
    dl_max_epochs=DL_MAX_EPOCHS, dl_batch_size=DL_BATCH_SIZE, cache_dir=CACHE,
), force=FORCE)
report_jitter_full(df_atcnet_jit, title=f"Full jitter — ATCNet — {DATASET}", modes=MODES)

## G — Leave-one-reference-out (LORO)

Hold out one reference at a time, train jitter over the rest, test on the
held-out one. The recovery gap is the cost of never training on that reference.
ShallowConvNet is primary.

In [ ]:
df_loro = run_or_skip(f"{DATASET}_shallow_loro_{TAG}", lambda: run_loro_matrix(
    DATASET, model="shallow", seeds=SEEDS, reference_modes=REFERENCES,
    dl_max_epochs=DL_MAX_EPOCHS, dl_batch_size=DL_BATCH_SIZE, cache_dir=CACHE,
), force=FORCE)
report_loro(df_loro, title=f"LORO — ShallowConvNet — {DATASET}", modes=MODES)

## H — Leave-one-family-out (LOFO)

Hold out a whole family (global / single / spatial), train jitter over the
others, test on every reference. Tests generalisation across *kinds* of
reference operation. ShallowConvNet is primary.

In [ ]:
df_lofo = run_or_skip(f"{DATASET}_shallow_lofo_{TAG}", lambda: run_lofo_matrix(
    DATASET, model="shallow", families=FAMILIES, seeds=SEEDS, reference_modes=REFERENCES,
    dl_max_epochs=DL_MAX_EPOCHS, dl_batch_size=DL_BATCH_SIZE, cache_dir=CACHE,
), force=FORCE)
report_lofo(df_lofo, title=f"LOFO — ShallowConvNet — {DATASET}")

## I — Summary: transfer gaps with CIs

The headline table. Each row is a model/condition's subject-level transfer gap
(matched − mismatched) with a bootstrap CI over subjects. Only runs already
cached this session appear.

In [ ]:
runs = [
    ("CSP+LDA",        "no EA", "df_csp_lda"),
    ("ShallowConvNet", "no EA", "df_shallow"),
    ("EEGNet",         "no EA", "df_eegnet"),
    ("ATCNet",         "no EA", "df_atcnet"),
    ("CSP+LDA",        "EA",    "df_csp_lda_ea"),
    ("ShallowConvNet", "EA",    "df_shallow_ea"),
]
rows = []
for model, cond, var in runs:
    df = globals().get(var)
    if df is None:
        continue
    r = transfer_gap_ci(df)
    rows.append({"model": model, "condition": cond, "gap_%": r["mean"],
                 "ci_lo": r["lo"], "ci_hi": r["hi"], "n_subj": r["n_subjects"]})
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
summary.to_csv(f"{RESULTS}/{DATASET}_transfer_gap_summary_{TAG}.csv", index=False)